# Parameterized CARL with Nuisance ParametersTrain a classifier to estimate the likelihood ratio$$r(x|\theta,\nu) = \frac{p(x|\theta,\nu)}{p(x|\theta_{\text{SM}}, \nu=0)}$$parameterized on both EFT parameters $\theta = (C_W/\Lambda^2,\; C_{\tilde{W}}/\Lambda^2)$and three nuisance parameters:| Nuisance | MadMiner type | Physical meaning ||----------|--------------|-----------------|| `mu_scale` | `scale`, `mu` | Correlated $\mu_R, \mu_F$ variation || `pdf_unc` | `norm`, 1.07 | PDF uncertainty (~7% from NNPDF replicas) || `signal_norm` | `norm`, 1.10 | Signal cross-section normalization |The network sees 5 parameter dimensions: $[\theta_1, \theta_2, \nu_\mu, \nu_{\text{pdf}}, \nu_{\text{norm}}]$.

## 0. Environment Setup

In [ ]:
import osos.environ["LD_LIBRARY_PATH"] = (    "/madminer/software/MG5_aMC_v2_9_16/HEPTools/lhapdf6_py3/lib:"    + os.environ.get("LD_LIBRARY_PATH", ""))

In [ ]:
import loggingimport numpy as npimport matplotlib.pyplot as pltfrom pathlib import Pathfrom madminer.core import MadMinerfrom madminer.lhe import LHEReaderfrom madminer.sampling import SampleAugmenterfrom madminer.analysis import DataAnalyzerfrom madminer.ml import ParameterizedRatioEstimatorfrom particle import Particlemg_dir = os.getenv("MG_FOLDER_PATH")logging.basicConfig(    format="%(asctime)-5.5s %(name)-20.20s %(levelname)-7.7s %(message)s",    datefmt="%H:%M",    level=logging.INFO,)for key in logging.Logger.manager.loggerDict:    if "madminer" not in key:        logging.getLogger(key).setLevel(logging.WARNING)STAGING_DIR = "/staging/jsandesara/madminer/semi_parametric_carl"os.makedirs(STAGING_DIR, exist_ok=True)DATA_DIR = "data/carl_nuisance"Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

## 1. Physics SetupSame EFT parameters and benchmarks as the main semi-parametric workflow.

In [ ]:
miner = MadMiner()miner.add_parameter(    lha_block="dim6", lha_id=2, parameter_name="CWL2",    morphing_max_power=2, param_card_transform="16.52*theta",    parameter_range=(-20.0, 20.0),)miner.add_parameter(    lha_block="dim6", lha_id=5, parameter_name="CPWL2",    morphing_max_power=2, param_card_transform="16.52*theta",    parameter_range=(-20.0, 20.0),)miner.add_benchmark({"CWL2": 0.0, "CPWL2": 0.0}, "sm")miner.add_benchmark({"CWL2": 5.0, "CPWL2": 0.0}, "w")miner.add_benchmark({"CWL2": -5.0, "CPWL2": 0.0}, "neg_w")miner.add_benchmark({"CWL2": 0.0, "CPWL2": 5.0}, "ww")miner.add_benchmark({"CWL2": 0.0, "CPWL2": -5.0}, "neg_ww")miner.set_morphing(include_existing_benchmarks=True, max_overall_power=2)

## 2. SystematicsThree nuisance parameters, each a single dimension:- **mu_scale**: correlated mu_R/mu_F scale variation (0.5x to 2x)- **pdf_unc**: PDF normalization uncertainty (~7%, from NNPDF23 replicas)- **signal_norm**: signal cross-section normalization (10%)NOTE:  is modeled as a flat normalization here. For shape-dependentPDF uncertainty, use  and extract replica weights.

In [ ]:
miner.add_systematics(effect="scale", systematic_name="mu_scale", scale="mu")miner.add_systematics(effect="norm", systematic_name="pdf_unc", norm_variation=1.07)miner.add_systematics(effect="norm", systematic_name="signal_norm", norm_variation=1.10)miner.save(f"{DATA_DIR}/setup.h5")print("Systematics:")for name, syst in miner.systematics.items():    print(f"  {name}: type={syst.type.value}, scale={syst.scale}, value={syst.value}")

## 3. Event GenerationOnly  needs MadGraph; norm systematics are applied analytically.

In [ ]:
mg_systematics = ["mu_scale"]all_systematics = ["mu_scale", "pdf_unc", "signal_norm"]benchmarks = ["sm", "w", "neg_w", "ww", "neg_ww"]for bm in benchmarks:    lhe_path = f"./mg_processes/carl_{bm}/Events/run_01/unweighted_events.lhe.gz"    if os.path.exists(lhe_path):        print(f"LHE for {bm} exists, skipping")        continue    miner.run(        sample_benchmark=bm,        mg_directory=mg_dir,        mg_process_directory=f"./mg_processes/carl_{bm}",        proc_card_file="cards/proc_card_signal.dat",        param_card_template_file="cards/param_card_template.dat",        run_card_file="cards/run_card_signal_small.dat",        log_directory="logs/carl",        systematics=mg_systematics,    )

## 4. Parse LHE, Define Observables, Analyse

In [ ]:
lhe = LHEReader(f"{DATA_DIR}/setup.h5")for bm in benchmarks:    lhe.add_sample(        lhe_filename=f"mg_processes/carl_{bm}/Events/run_01/unweighted_events.lhe.gz",        sampled_from_benchmark=bm,        is_background=False,        k_factor=1.1,        systematics=all_systematics,    )particles = [    *Particle.findall(lambda p: p.pdgid.is_quark),    *Particle.findall(pdg_name="g"),]lhe.set_smearing(    pdgids=[int(p.pdgid) for p in particles],    energy_resolution_abs=0.0, energy_resolution_rel=0.1,    eta_resolution_abs=0.1, eta_resolution_rel=0.0,    phi_resolution_abs=0.1, phi_resolution_rel=0.0,)lhe.add_observable("pt_j1",  "j[0].pt",  required=True)lhe.add_observable("eta_j1", "j[0].eta", required=True)lhe.add_observable("phi_j1", "j[0].phi", required=True)lhe.add_observable("e_j1",   "j[0].e",   required=True)lhe.add_observable("pt_j2",  "j[1].pt",  required=True)lhe.add_observable("eta_j2", "j[1].eta", required=True)lhe.add_observable("phi_j2", "j[1].phi", required=True)lhe.add_observable("e_j2",   "j[1].e",   required=True)lhe.add_observable("m_jj",         "(j[0]+j[1]).m",       required=True)lhe.add_observable("delta_eta_jj", "j[0].deltaeta(j[1])", required=True)lhe.add_observable("delta_phi_jj",    "j[0].deltaphi(j[1]) * (-1.0 + 2.0 * float(j[0].eta > j[1].eta))",    required=True)lhe.add_observable("pt_a1",  "a[0].pt",  required=True)lhe.add_observable("eta_a1", "a[0].eta", required=True)lhe.add_observable("phi_a1", "a[0].phi", required=True)lhe.add_observable("e_a1",   "a[0].e",   required=True)lhe.add_observable("pt_a2",  "a[1].pt",  required=True)lhe.add_observable("eta_a2", "a[1].eta", required=True)lhe.add_observable("phi_a2", "a[1].phi", required=True)lhe.add_observable("e_a2",   "a[1].e",   required=True)lhe.add_observable("m_aa",  "(a[0]+a[1]).m",  required=True)lhe.add_observable("pt_aa", "(a[0]+a[1]).pt", required=True)lhe.add_observable("met", "met.pt", required=True)lhe.add_cut("(a[0] + a[1]).m > 122.0")lhe.add_cut("(a[0] + a[1]).m < 128.0")lhe.add_cut("pt_j1 > 30.0")lhe.analyse_samples()lhe_data_path = f"{STAGING_DIR}/lhe_data_carl.h5"lhe.save(lhe_data_path)print(f"Saved to {lhe_data_path}")

## 5. Inspect

In [ ]:
da = DataAnalyzer(lhe_data_path, include_nuisance_parameters=True)print(f"Parameters:   {list(da.parameters.keys())}")print(f"Nuisance:     {list(da.nuisance_parameters.keys())}")print(f"Systematics:  {list(da.systematics.keys())}")print(f"Benchmarks:   {list(da.benchmarks.keys())}")print(f"Morphing:     {da.morpher is not None}")print(f"Nuis morpher: {da.nuisance_morpher is not None}")x, w = da.weighted_events(theta=None)print(f"Events: {x.shape[0]}, Weights: {w.shape}")

## 6. Sample Training DataALICES method: the sampler produces (x, y, theta, r_xz, t_xz). is automatically  (5D).

In [ ]:
sampler = SampleAugmenter(lhe_data_path)n_nuis = len(sampler.nuisance_parameters)print(f"Nuisance params ({n_nuis}): {list(sampler.nuisance_parameters.keys())}")samples_dir = f"{DATA_DIR}/samples"Path(samples_dir).mkdir(parents=True, exist_ok=True)theta0 = ("random_morphing_points", (None, [("flat", -15, 15), ("flat", -15, 15)]))theta1 = ("benchmark", "sm")nu0 = ("random_morphing_points", (None, [("gaussian", 0, 1)] * n_nuis))x, theta0_out, t_xz, _ = sampler.sample_train_ratio(    theta0=theta0, theta1=theta1,    nu0=nu0, nu1=None,    n_samples=100_000,    nuisance_score=True,    folder=samples_dir, filename="train",)print(f"theta shape: {np.load(f'{samples_dir}/theta0_train.npy').shape}")print("columns: [CWL2, CPWL2, nu_mu, nu_pdf, nu_norm]")

## 7. Train

In [ ]:
estimator = ParameterizedRatioEstimator(n_hidden=(256, 256, 256), activation="tanh")losses_train, losses_val = estimator.train(    method="alices",    x=f"{samples_dir}/x_train.npy",    y=f"{samples_dir}/y_train.npy",    theta=f"{samples_dir}/theta0_train.npy",    r_xz=f"{samples_dir}/r_xz_train.npy",    t_xz=f"{samples_dir}/t_xz_train.npy",    n_epochs=50, batch_size=256,    initial_lr=1e-3, final_lr=1e-5,    validation_split=0.25,    early_stopping=True, early_stopping_patience=10,)estimator.save(f"{DATA_DIR}/models/carl_3nuis")

## 8. Training Loss

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))ax.plot(losses_train, label="Train")ax.plot(losses_val, label="Validation")ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")ax.set_title("ALICES (EFT + 3 Nuisance)")ax.legend(); ax.set_yscale("log")plt.tight_layout()plt.savefig(f"{DATA_DIR}/training_loss.png", dpi=150)plt.show()

## 9. Closure: Predicted vs True

In [ ]:
model = ParameterizedRatioEstimator()model.load(f"{DATA_DIR}/models/carl_3nuis")x_test = np.load(f"{samples_dir}/x_test.npy")theta_test = np.load(f"{samples_dir}/theta0_test.npy")r_xz_test = np.load(f"{samples_dir}/r_xz_test.npy")log_r_hat, _ = model.evaluate_log_likelihood_ratio(x=x_test, theta=theta_test)log_r_true = np.log(r_xz_test.flatten() + 1e-30)log_r_hat = log_r_hat.flatten()fig, axes = plt.subplots(1, 2, figsize=(14, 5))finite = np.isfinite(log_r_true) & np.isfinite(log_r_hat)ax = axes[0]ax.scatter(log_r_true[finite], log_r_hat[finite], alpha=0.02, s=2, rasterized=True)lims = [max(log_r_true[finite].min(), -15), min(log_r_true[finite].max(), 15)]ax.plot(lims, lims, "r--", lw=2)ax.set_xlabel(r"True $\log r$"); ax.set_ylabel(r"Predicted $\log \hat{r}$")ax.set_title("Closure"); ax.set_xlim(lims); ax.set_ylim(lims)ax = axes[1]res = (log_r_hat - log_r_true)[finite]ax.hist(res, bins=100, density=True)ax.set_xlabel(r"$\log\hat{r} - \log r$"); ax.set_ylabel("Density")ax.set_title(f"Residuals: mean={res.mean():.3f}, std={res.std():.3f}")ax.axvline(0, color="r", ls="--")plt.tight_layout()plt.savefig(f"{DATA_DIR}/closure.png", dpi=150)plt.show()

## 10. Nuisance Parameter ImpactScan each nuisance at a fixed EFT point to see the learned dependence.

In [ ]:
y_test = np.load(f"{samples_dir}/y_test.npy")sm_mask = (y_test.flatten() == 1)x_sm = x_test[sm_mask][:500]theta_fixed = np.array([10.0, 0.0, 0.0, 0.0, 0.0])nu_scan = np.linspace(-2, 2, 41)nuis_labels = [r"$u_\mu$ (scale)", r"$u_{\mathrm{pdf}}$", r"$u_{\mathrm{norm}}$"]nuis_indices = [2, 3, 4]fig, axes = plt.subplots(1, 3, figsize=(18, 5))for ax, idx, label in zip(axes, nuis_indices, nuis_labels):    mean_lr = []    for v in nu_scan:        th = np.tile(theta_fixed, (len(x_sm), 1))        th[:, idx] = v        lr, _ = model.evaluate_log_likelihood_ratio(x=x_sm, theta=th)        mean_lr.append(lr.mean())    ax.plot(nu_scan, mean_lr, "b-", lw=2)    ax.axvline(0, color="gray", ls="--", alpha=0.5)    ax.set_xlabel(label); ax.set_ylabel(r"$\langle \log\hat{r} angle$")    ax.set_title(f"Impact of {label}")plt.suptitle(r"$C_W/\Lambda^2 = 10,\; C_{	ilde{W}}/\Lambda^2 = 0$", y=1.02)plt.tight_layout()plt.savefig(f"{DATA_DIR}/nuisance_impact.png", dpi=150)plt.show()

## 11. Profiled vs Fixed NLLCompare NLL with nuisance fixed at nominal vs profiled (minimized).

In [ ]:
from scipy.optimize import minimizen_params = 2n_nuis = 3n_scan = 25cwl2_v = np.linspace(-15, 15, n_scan)cpwl2_v = np.linspace(-15, 15, n_scan)cg, cpg = np.meshgrid(cwl2_v, cpwl2_v)x_eval = x_sm[:200]nll_fixed = np.zeros((n_scan, n_scan))nll_prof = np.zeros((n_scan, n_scan))def nll(theta_full):    th = np.tile(theta_full, (len(x_eval), 1))    lr, _ = model.evaluate_log_likelihood_ratio(x=x_eval, theta=th)    return -np.mean(lr)for i in range(n_scan):    for j in range(n_scan):        base = np.array([cg[i,j], cpg[i,j], 0., 0., 0.])        nll_fixed[i,j] = nll(base)        res = minimize(lambda nu: nll(np.concatenate([base[:n_params], nu])),                       x0=np.zeros(n_nuis), method="L-BFGS-B",                       bounds=[(-3,3)]*n_nuis)        nll_prof[i,j] = res.funnll_fixed -= nll_fixed.min()nll_prof -= nll_prof.min()fig, axes = plt.subplots(1, 2, figsize=(14, 5))for ax, data, title in [(axes[0], nll_fixed, r"Fixed $u=0$"),                         (axes[1], nll_prof, r"Profiled over $u$")]:    c = ax.contourf(cg, cpg, data, levels=30, cmap="viridis")    plt.colorbar(c, ax=ax)    ax.contour(cg, cpg, data, levels=[2.30, 5.99], colors=["white","lightgray"], linewidths=2)    ax.plot(0, 0, "w*", ms=15)    ax.set_xlabel(r"$C_W/\Lambda^2$"); ax.set_ylabel(r"$C_{	ilde{W}}/\Lambda^2$")    ax.set_title(title)plt.tight_layout()plt.savefig(f"{DATA_DIR}/nll_2d.png", dpi=150)plt.show()print("Done!")